# Tier 2 — the Unknown branch: where is the record too weak to trust?

Tier 1 ([`06_analysis.ipynb`](06_analysis.ipynb)) predicts an Unknown share — burned area with **no
attributed cause**, 18.5% of all acres — as a class in its own right. This branch treats that mass as
a **data-quality signal** rather than a fire forecast, and asks two questions:

1. Can a region-season's `missing_acre_frac` be predicted from its own history, so a planner is
   warned before the season that attribution will be weak there?
2. Across ecoregions, does missing share rise with Natural share?

**Input.** `data/region_season_cause.parquet` — EPA Level III ecoregion × meteorological season ×
season-year, with the missing-cause mass in `missing_acres`.

**Deliverable.** A targeting list: region-seasons where attribution is weak and the burned area is
material, ranked by predicted unattributed acres.

Same grain, partial-winter boundary rule, and forward-chaining split (score season-year ≥ 2010) as
the rest of the pipeline.

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from config import ProjectConfig
from panel import RegionSeasonPanel

cfg = ProjectConfig()

# `attribution_quality()` gives one row per region-season: the missing-cause fraction plus the
# coarse Human/Natural acres, so data quality can be related to the Natural share. The per-cell
# missing_* columns are DEDUPLICATED, not summed -- they repeat across the cell's 12 cause rows,
# and summing would inflate the Unknown mass 12x.
panel = RegionSeasonPanel.load(cfg)
cell = panel.attribution_quality()

print(f"{len(cell):,} region-season cells")
print(f"missing_acre_frac: mean {cell.missing_acre_frac.mean():.3f}, "
      f"median {cell.missing_acre_frac.median():.3f}")

10,135 region-season cells
missing_acre_frac: mean 0.238, median 0.118


## Can next season's attribution quality be predicted in advance?

**Hypothesis.** A region-season's missing-cause fraction is stable year to year, so its own trailing
history predicts it better than a single national constant does.

**Experiment.** Predict `missing_acre_frac` from the trailing mean of the cell's prior same-season
occurrences (k=7), scored on the held-out tail (season-year ≥ 2010) against a global-mean baseline
fit on training years only. `missing_acre_frac` is a bounded [0,1] fraction, so the metric is MAE in
raw fraction points — no log transform.

In [2]:
TEST_START = cfg.test_start     # forward-chaining split, shared across all branches
K = cfg.shares_k                # trailing window locked by the Tier-1 shares sweep

# TrailingMean does shift(1) then a k-window mean within (region, season), and asserts the frame
# is sorted first -- on an unsorted frame the raw idiom silently attaches one region's history to
# another's rows.
from trailing import GlobalPrior, TrailingMean

pred_trail = TrailingMean(K).predict(cell, "missing_acre_frac")["missing_acre_frac"].to_numpy()

actual = cell["missing_acre_frac"].to_numpy()
w = cell["total_ac"].to_numpy()
in_test = (cell["season_year"] >= TEST_START).to_numpy()
train = (cell["season_year"] < TEST_START).to_numpy()

def mae(pred):
    err = np.abs(pred - actual)
    m = in_test & ~np.isnan(err)
    return {"n_cells": int(m.sum()),
            "MAE_unwtd": float(err[m].mean()),
            "MAE_acre_wtd": float(np.average(err[m], weights=w[m]))}

# Reference: one constant for every cell, fit on training years only (no region info).
pred_global = (GlobalPrior(weighted=False)
               .fit(cell, "missing_acre_frac", train_mask=train)
               .predict(cell)["missing_acre_frac"].to_numpy())

res_q1 = pd.DataFrame([mae(pred_global), mae(pred_trail)],
                      index=["global mean", f"persistence (k={K})"])
print(f"missing_acre_frac -- held-out tail season_year >= {TEST_START}\n")
print(res_q1.round(4).to_string())

missing_acre_frac -- held-out tail season_year >= 2010

                   n_cells  MAE_unwtd  MAE_acre_wtd
global mean           3954     0.2230        0.2398
persistence (k=7)     3949     0.2012        0.1665


**Finding.** Attribution quality is forecastable. Persistence beats the global mean on both metrics —
MAE 0.201 vs 0.223 unweighted, and 0.167 vs 0.240 acre-weighted. The acre-weighted gap is the wider
one, so the cells carrying the most burned area are the ones persistence predicts best.

A region-season's attribution weakness is therefore a property of the region-season, not a national
constant, and the planner can be warned before the season rather than after.

## Does missingness concentrate in the high-Natural West?

**Hypothesis.** If unattributed acres were would-be Natural fires going unlabelled, missing share
would rise with Natural share across ecoregions — a positive correlation.

**Experiment.** Aggregate to the ecoregion (the claim is about regions, not individual cells): total
Natural, Human and missing acres over the full record, then correlate Natural share against missing
share across the 105 ecoregions.

This is a cross-sectional **level** test. [`03_missingness.ipynb`](03_missingness.ipynb) tested the
same question from three other angles — detrended over time, within region×season, and by fire size
— and found Natural flat (r ≈ −0.02 detrended, ≈ +0.02 within-cell), with the one real
size-selectivity landing on small human/Debris fires. This is an independent fourth angle on the
same question.

In [3]:
# Total Natural / Human / missing acres per ecoregion over the full record, then shares.
reg = cell.groupby("region").agg(nat=("Natural", "sum"), hum=("Human", "sum"),
                                 miss=("missing_acres", "sum")).reset_index()
reg["tot"] = reg["nat"] + reg["hum"] + reg["miss"]
reg = reg[reg["tot"] > 0].copy()
reg["nat_share"] = reg["nat"] / reg["tot"]
reg["miss_share"] = reg["miss"] / reg["tot"]

pear = reg["nat_share"].corr(reg["miss_share"])
spear = reg["nat_share"].corr(reg["miss_share"], method="spearman")
print(f"across {len(reg)} ecoregions -- corr(Natural share, missing share):")
print(f"   Pearson  {pear:+.3f}")
print(f"   Spearman {spear:+.3f}\n")

print("Highest-missing ecoregions (where attribution is weakest):")
print(reg.nlargest(8, "miss_share")[["region", "miss_share", "nat_share"]].round(2).to_string(index=False))
print("\nLowest-missing ecoregions:")
print(reg.nsmallest(8, "miss_share")[["region", "miss_share", "nat_share"]].round(2).to_string(index=False))

across 105 ecoregions -- corr(Natural share, missing share):
   Pearson  -0.636
   Spearman -0.558

Highest-missing ecoregions (where attribution is weakest):
                  region  miss_share  nat_share
    Central Great Plains        0.69       0.05
   Southern Texas Plains        0.64       0.01
 Southwestern Tablelands        0.63       0.20
             Flint Hills        0.59       0.02
         Edwards Plateau        0.58       0.19
             Coast Range        0.53       0.27
             High Plains        0.53       0.20
Eastern Corn Belt Plains        0.52       0.01

Lowest-missing ecoregions:
                  region  miss_share  nat_share
        Aleutian Islands         0.0       0.00
    Arctic Coastal Plain         0.0       0.99
       Ogilvie Mountains         0.0       1.00
      Wrangell Mountains         0.0       1.00
        Arctic Foothills         0.0       1.00
             Yukon Flats         0.0       1.00
            Brooks Range         0.0       0.

**Finding.** The correlation is strongly **negative** — Pearson −0.64, Spearman −0.56 across 105
ecoregions. Missingness concentrates in low-Natural, human-dominated regions, not the high-Natural
West.

The extremes are unambiguous. Highest-missing: Central Great Plains (69% missing, 5% Natural),
Southern Texas Plains (64%, 1%), Flint Hills (59%, 2%), Edwards Plateau (58%, 19%). Lowest-missing:
the Alaskan and Arctic ecoregions — Ogilvie Mountains, Yukon Flats, Brooks Range, Arctic Foothills —
all at ~0% missing and 99–100% Natural.

**This makes the resolved Human 22.7% a floor.** Unattributed acres sit disproportionately in human
regions, so redistributing any of the Unknown mass onto resolved causes would add more to Human than
to Natural. The true Human share is if anything higher than 22.7%. Tier 1 keeps that floor visible
by predicting the Unknown share as its own class rather than distributing it.

The pattern is consistent with non-attribution being an agency and reporting-practice axis in
human-dominated country — the high-missing states (NY, CO, KS, AZ, CA) and the BLM / ST&L
reporting-stream rise characterized in [`03_missingness.ipynb`](03_missingness.ipynb) — rather than
Natural fire being relabelled in the West.

## Where should the planner invest in cause reporting?

Rank held-out region-seasons by predicted unattributed acres — the product of predicted weakness
(`missing_acre_frac`) and acres at stake. A region-season qualifies only if attribution is both weak
and the burn is material; either alone is not worth acting on.

In [4]:
# Predicted missing acres = predicted missing fraction x total acres, on scored held-out cells.
scored = in_test & ~np.isnan(pred_trail)
out = cell[scored].copy()
out["pred_missing_frac"] = pred_trail[scored]
out["pred_missing_ac"] = out["pred_missing_frac"] * out["total_ac"]

# Aggregate over the tail to a region-season targeting list (mean predicted weakness + acres at stake).
target = (out.groupby(["region", "season"])
          .agg(mean_pred_missing_frac=("pred_missing_frac", "mean"),
               total_acres_at_stake=("total_ac", "sum"),
               pred_missing_acres=("pred_missing_ac", "sum"))
          .reset_index())

print("Where to invest in cause reporting -- top region-seasons by predicted unattributed acres")
print("(weak attribution AND material burn), held-out tail:\n")
print(target.nlargest(12, "pred_missing_acres")
      .assign(mean_pred_missing_frac=lambda d: (d.mean_pred_missing_frac * 100).round(0),
              total_acres_at_stake=lambda d: (d.total_acres_at_stake / 1e3).round(0),
              pred_missing_acres=lambda d: (d.pred_missing_acres / 1e3).round(0))
      .rename(columns={"mean_pred_missing_frac": "miss%", "total_acres_at_stake": "acres_k",
                       "pred_missing_acres": "pred_miss_k"})
      .to_string(index=False))

Where to invest in cause reporting -- top region-seasons by predicted unattributed acres
(weak attribution AND material burn), held-out tail:

                                             region season  miss%  acres_k  pred_miss_k
                            Southwestern Tablelands    MAM   46.0   2495.0       1165.0
 Central California Foothills and Coastal Mountains    JJA   36.0   2463.0        851.0
                                   Columbia Plateau    JJA   38.0   2107.0        831.0
                               Central Great Plains    MAM   57.0   1151.0        703.0
                                   Columbia Plateau    SON   59.0    796.0        538.0
                                        High Plains    MAM   43.0   1028.0        441.0
Klamath Mountains/California High North Coast Range    JJA   17.0   3006.0        415.0
                            Central Basin and Range    JJA   11.0   3297.0        395.0
                                     North Cascades    JJA   31.0

**Finding.** The list is headed by Southwestern Tablelands MAM — 46% predicted missing on 2.5M acres,
1.17M unattributed acres — followed by Central California Foothills JJA (851k) and Columbia Plateau
JJA (831k).

Two distinct kinds of entry appear, and they warrant different responses. Some are weak-attribution
cells: Central Great Plains MAM (57% missing) and Columbia Plateau SON (59%) rank high because most
of their burn is unattributed, not because they burn most. Others are high-volume cells where even a
modest missing rate is material: Central Basin and Range JJA is only 11% missing but burns 3.3M
acres, and Klamath Mountains JJA is 17% on 3.0M.